# RetailPartnerX — Shopping Assistant (hw-3)

Минимальный прототип мультиагентной системы на **LangGraph**.

**Сценарий:** «Собери ужин на 4 человека: паста, без глютена, бюджет до 2000 ₽»

| Агент | Роль |
|-------|------|
| **Orchestrator** | Менеджер: планирует шаги, делегирует, собирает ответ |
| **Catalog Searcher** | Поисковик SKU в mock-каталоге |
| **Policy RAG (stub)** | Проверка ограничений по Policy KB |

> Демо работает **без API-ключей** — rule-based парсер и keyword retrieval.

In [ ]:
!pip install -q langgraph langchain-core

In [ ]:
from __future__ import annotations

import re
from typing import TypedDict

import pandas as pd
from langgraph.graph import END, StateGraph

USER_QUERY = "Собери ужин на 4 человека: паста, без глютена, бюджет до 2000 рублей"

CATALOG = pd.DataFrame([
    {"sku": "SKU-101", "title": "Паста рисовая без глютена", "category": "паста", "price": 189, "gluten_free": True},
    {"sku": "SKU-102", "title": "Соус томатный классический", "category": "соусы", "price": 129, "gluten_free": True},
    {"sku": "SKU-103", "title": "Пармезан тёртый", "category": "сыры", "price": 249, "gluten_free": True},
    {"sku": "SKU-104", "title": "Паста пшеничная спагетти", "category": "паста", "price": 99, "gluten_free": False},
    {"sku": "SKU-105", "title": "Оливковое масло Extra Virgin", "category": "масла", "price": 399, "gluten_free": True},
    {"sku": "SKU-106", "title": "Салат руккола", "category": "овощи", "price": 159, "gluten_free": True},
    {"sku": "SKU-107", "title": "Крутоны пшеничные", "category": "готовые", "price": 89, "gluten_free": False},
    {"sku": "SKU-108", "title": "Соус песто", "category": "соусы", "price": 219, "gluten_free": True},
])

POLICY_KB = [
    {"policy_id": "POL-ALLERGEN-01", "text": "Товары с глютеном запрещены при запросе безглютенового рациона."},
    {"policy_id": "POL-ALLERGEN-02", "text": "Паста из пшеницы содержит глютен и не подходит для диеты gluten-free."},
    {"policy_id": "POL-PROMO-01", "text": "Акция 2+1 на соусы томатные до конца месяца."},
    {"policy_id": "POL-GDPR-01", "text": "Персональные рекомендации требуют согласия пользователя на обработку данных."},
]

MESSAGES: list[str] = []


def log(agent: str, text: str) -> None:
    line = f"[{agent}] {text}"
    MESSAGES.append(line)
    print(line)

In [ ]:
class AgentState(TypedDict):
    user_query: str
    constraints: dict
    policy_chunks: list[dict]
    policy_verdict: str
    candidates: list[dict]
    final_answer: str


def parse_constraints(query: str) -> dict:
  persons = 4
  m = re.search(r"(\d+)\s*челов", query.lower())
  if m:
    persons = int(m.group(1))
  budget = 2000
  m = re.search(r"(\d+)\s*(?:₽|руб)", query.lower())
  if m:
    budget = int(m.group(1))
  return {
    "dish": "паста" if "паст" in query.lower() else "ужин",
    "persons": persons,
    "budget": budget,
    "gluten_free": "без глютена" in query.lower() or "gluten" in query.lower(),
  }


def policy_rag_retrieve(query: str, top_k: int = 2) -> list[dict]:
  tokens = set(re.findall(r"[а-яa-z0-9]+", query.lower()))
  scored = []
  for chunk in POLICY_KB:
    words = set(re.findall(r"[а-яa-z0-9]+", chunk["text"].lower()))
    score = len(tokens & words)
    if "глютен" in query.lower() and "глютен" in chunk["text"].lower():
      score += 2
    scored.append((score, chunk))
  scored.sort(key=lambda x: x[0], reverse=True)
  return [c for s, c in scored[:top_k] if s > 0] or POLICY_KB[:2]


def orchestrator_node(state: AgentState) -> AgentState:
  log("Orchestrator", f"Получен запрос: {state['user_query']}")
  constraints = parse_constraints(state["user_query"])
  log("Orchestrator", f"Извлечены ограничения: {constraints}")
  chunks = policy_rag_retrieve(state["user_query"])
  log("Orchestrator", f"Делегирую Policy RAG, chunks={[c['policy_id'] for c in chunks]}")
  return {**state, "constraints": constraints, "policy_chunks": chunks}


def policy_analyst_node(state: AgentState) -> AgentState:
  verdict = "allow"
  if state["constraints"].get("gluten_free"):
    verdict = "allow"  # фильтрация на этапе поиска
  log("PolicyAnalyst", f"RAG verdict={verdict}, citations={[c['policy_id'] for c in state['policy_chunks']]}")
  return {**state, "policy_verdict": verdict}


def catalog_searcher_node(state: AgentState) -> AgentState:
  c = state["constraints"]
  log("CatalogSearcher", f"Ищу SKU: dish={c['dish']}, budget={c['budget']}, gluten_free={c['gluten_free']}")
  df = CATALOG.copy()
  if c["gluten_free"]:
    df = df[df["gluten_free"]]
  if c["dish"]:
    df = df[df["category"].str.contains(c["dish"], case=False) | df["title"].str.contains(c["dish"], case=False)]
  df = df[df["price"] <= c["budget"] / max(c["persons"], 1)]
  candidates = df.head(3).to_dict("records")
  log("CatalogSearcher", f"Найдено {len(candidates)} SKU: {[x['sku'] for x in candidates]}")
  return {**state, "candidates": candidates}


def synthesize_node(state: AgentState) -> AgentState:
  if state["policy_verdict"] != "allow":
    answer = "Не могу подобрать товары: нарушение политики компании."
  elif not state["candidates"]:
    answer = "Подходящих товаров не найдено. Уточните бюджет или ограничения."
  else:
    lines = [f"• {p['title']} ({p['sku']}) — {p['price']} ₽" for p in state["candidates"]]
    answer = "Корзина для ужина без глютена:\n" + "\n".join(lines)
  log("Orchestrator", "Синтез финального ответа")
  log("Orchestrator", answer.replace("\n", " | "))
  return {**state, "final_answer": answer}

In [ ]:
graph = StateGraph(AgentState)
graph.add_node("orchestrator", orchestrator_node)
graph.add_node("policy_analyst", policy_analyst_node)
graph.add_node("catalog_searcher", catalog_searcher_node)
graph.add_node("synthesize", synthesize_node)

graph.set_entry_point("orchestrator")
graph.add_edge("orchestrator", "policy_analyst")
graph.add_edge("policy_analyst", "catalog_searcher")
graph.add_edge("catalog_searcher", "synthesize")
graph.add_edge("synthesize", END)

app = graph.compile()
result = app.invoke({"user_query": USER_QUERY})

print("\n=== FINAL ANSWER ===")
print(result["final_answer"])
print("\n=== MESSAGE LOG ===")
for msg in MESSAGES:
    print(msg)